In [1]:
import pandas as pd
import re
from tqdm.notebook import tqdm  # or from tqdm import tqdm
import numpy as np

In [39]:
FILE_CLINICAL_LINKING = "/scratch/sdonev/Clinical_Pipeline/data/linked_to_ontologies/entities_drug_disease_clin.csv"


In [40]:
clinical_df = pd.read_csv(FILE_CLINICAL_LINKING)

In [41]:
clinical_df.head()

,nct_id,merged_condition_names,disease_mondo_termid,disease_mondo_term_norm,disease_term_mondo_clean,disease_termid_mondo_clean,nearest_dataset_parent_mondo,nearest_dataset_parent_label,merged_mondo_termid,merged_mondo_label,ner_predicted_drugs,linkbert_umls_drugs,drug_umls_termid,drug_umls_term_norm,nearest_dataset_parent_umls,nearest_dataset_parent_umls_label,merged_umls_termid,merged_umls_label
0,NCT00000114,Retinitis Pigmentosa,MONDO:0008377,retinitis pigmentosa 1,retinitis pigmentosa,MONDO:0008377,-1,-1,MONDO:0008377,retinitis pigmentosa,vitamin a | vitamin e,Vitamin A|Vitamin E,C0042839|C0042874,Vitamin A|Vitamin E,-1,-1,C0042839|C0042874,Vitamin A|Vitamin E
1,NCT00000102,Congenital Adrenal Hyperplasia,MONDO:0015898,adrenogenital syndrome,adrenogenital syndrome,MONDO:0015898,-1,-1,MONDO:0015898,adrenogenital syndrome,glucocorticoid | nifedipine | procardia,glucocorticoid|Nifedipine|Procardia,C0017710|C0028066|C0700861,GLUCOCORTICOIDS|Nifedipine|Procardia,C0028066,Nifedipine,C0017710|C0028066|C0700861,GLUCOCORTICOIDS|Nifedipine|Procardia
2,NCT00000115,"Macular Edema, Cystoid | cystoid macular edema...",MONDO:0007935|MONDO:0007935|-1,cystoid macular edema|cystoid macular edema|uv...,cystoid macular edema|uveitis-associated,MONDO:0007935|-1,MONDO:0003005,macular retinal edema,MONDO:0007935|-1|MONDO:0003005,cystoid macular edema|uveitis-associated|macul...,acetazolamide,Acetazolamide,C0000981,Acetazolamide,-1,-1,C0000981,Acetazolamide
3,NCT00000117,Optic Neuritis | multiple sclerosis,MONDO:0005885|MONDO:0005301,optic neuritis|multiple sclerosis,optic neuritis|multiple sclerosis,MONDO:0005885|MONDO:0005301,-1|-1,-1|-1,MONDO:0005885|MONDO:0005301,optic neuritis|multiple sclerosis,intravenous immunoglobulin | ivig,intravenous immunoglobulin|ivig,C0085297|C0085297,Intravenous Immunoglobulins|Intravenous Immuno...,-1,-1,C0085297,Intravenous Immunoglobulins
4,NCT00000122,Glaucoma | filtering surgery | standard glauco...,MONDO:0005041|-1|-1,glaucoma|filtering surgery|standard glaucoma f...,glaucoma|filtering surgery|standard glaucoma f...,MONDO:0005041|-1|-1,-1,-1,MONDO:0005041|-1|-1,glaucoma|filtering surgery|standard glaucoma f...,5-fluorouracil,5-fluorouracil,C0016360,Fluorouracil,-1,-1,C0016360,Fluorouracil


In [42]:
neuro = pd.read_csv("./data/list_neuro_conditions.csv")
disease_terms = (
    neuro["disease_mondo_term_norm"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)

terms_set = set(disease_terms)
terms_list = list(terms_set)

In [58]:
terms = sorted({
    t.strip().lower()
    for t in terms_list
    if isinstance(t, str) and t.strip()
}, key=len, reverse=True)

terms = [t for t in terms if len(t) > 4]
pattern = re.compile("|".join(map(re.escape, terms)))


In [61]:
len(terms)

9688

In [62]:
s = (
    clinical_df["merged_mondo_label"]
    .fillna("")
    .astype(str)
    .str.lower()
)

terms_l = [t.lower() for t in terms]  # ensure terms are lowercase too

matched = np.full(len(s), None, dtype=object)
mask = np.zeros(len(s), dtype=bool)

chunk_size = 500

for i in tqdm(range(0, len(terms_l), chunk_size)):
    chunk_terms = terms_l[i : i + chunk_size]

    # Build alternation regex for this chunk
    pattern_str = "|".join(map(re.escape, chunk_terms))

    # Extract first match from this chunk (per row); NaN if none
    m = s.str.extract(f"({pattern_str})", expand=False)

    hit = m.notna().to_numpy()

    # update overall mask
    mask |= hit

    # only fill matched term where we don't already have one
    fill_idx = hit & (matched == None)
    matched[fill_idx] = m.to_numpy()[fill_idx]

clinical_df["matched_term"] = matched
clinical_neuro = clinical_df[mask]

clinical_neuro.shape, clinical_neuro["nct_id"].nunique()

  0%|          | 0/20 [00:00<?, ?it/s]

((48095, 19), 48095)

In [75]:
clinical_neuro[
    clinical_neuro['merged_mondo_label'].str.contains("xerostomia", case=False, na=False)
]

,nct_id,merged_condition_names,disease_mondo_termid,disease_mondo_term_norm,disease_term_mondo_clean,disease_termid_mondo_clean,nearest_dataset_parent_mondo,nearest_dataset_parent_label,merged_mondo_termid,merged_mondo_label,ner_predicted_drugs,linkbert_umls_drugs,drug_umls_termid,drug_umls_term_norm,nearest_dataset_parent_umls,nearest_dataset_parent_umls_label,merged_umls_termid,merged_umls_label,matched_term
957,NCT00001598,Lacrimal Apparatus Disease | Salivary Gland Di...,MONDO:0001793|MONDO:0001142|MONDO:0010030|-1|-...,excessive tearing|salivary gland disorder|Sjog...,excessive tearing|salivary gland disorder|Sjog...,MONDO:0001793|MONDO:0001142|MONDO:0010030|-1|-...,-1|-1|-1|MONDO:0000837|-1|-1,-1|-1|-1|bone resorption disease|-1|-1,MONDO:0001793|MONDO:0001142|MONDO:0010030|-1|-...,excessive tearing|salivary gland disorder|Sjog...,dehydroepiandrosterone | male hormone,dehydroepiandrosterone|male hormone,C0011185|-1,Prasterone|male hormone,-1,-1,C0011185|-1,Prasterone|male hormone,sjogren syndrome
961,NCT00001599,Sjogren's Syndrome | Xerostomia | dry eyes | d...,MONDO:0010030|-1|MONDO:0006733|-1,Sjogren syndrome|Xerostomia|dry eye syndrome|d...,Sjogren syndrome|Xerostomia|dry eye syndrome|d...,MONDO:0010030|-1|MONDO:0006733|-1,-1|-1,-1|-1,MONDO:0010030|-1|MONDO:0006733|-1,Sjogren syndrome|Xerostomia|dry eye syndrome|d...,thalidomide,Thalidomide,C0039736,Thalidomide,-1,-1,C0039736,Thalidomide,sjogren syndrome
14948,NCT00233363,Sjogren's Syndrome | Xerostomia | dry mouth,MONDO:0010030|-1|-1,Sjogren syndrome|Xerostomia|dry mouth,Sjogren syndrome|Xerostomia|dry mouth,MONDO:0010030|-1|-1,-1,-1,MONDO:0010030|-1|-1,Sjogren syndrome|Xerostomia|dry mouth,rebamipide,rebamipide,C0069562,rebamipide,-1,-1,C0069562,rebamipide,sjogren syndrome
24285,NCT00426543,Fatigue | Hyposalivation | Keratoconjunctiviti...,-1|-1|MONDO:0006733|MONDO:0010030|-1|MONDO:001...,Fatigue|Hyposalivation|dry eye syndrome|Sjogre...,Fatigue|Hyposalivation|dry eye syndrome|Sjogre...,-1|-1|MONDO:0006733|MONDO:0010030|-1,-1|-1,-1|-1,-1|-1|MONDO:0006733|MONDO:0010030|-1,Fatigue|Hyposalivation|dry eye syndrome|Sjogre...,"MabThera (rituximab) | Rituximab, Mabthera",riTUXimab|riTUXimab,C0393022|C0393022,riTUXimab|riTUXimab,-1,-1,C0393022,riTUXimab,sjogren syndrome
24831,NCT00438048,Primary Sjogren | Secondary Sjogren | Xerostom...,MONDO:0010030|MONDO:0010030|-1|MONDO:0006733|-...,Sjogren syndrome|Sjogren syndrome|Xerostomia|d...,Sjogren syndrome|Xerostomia|dry eye syndrome|d...,MONDO:0010030|-1|MONDO:0006733|-1|MONDO:0010434,-1|-1|-1,-1|-1|-1,MONDO:0010030|-1|MONDO:0006733|-1|MONDO:0010434,Sjogren syndrome|Xerostomia|dry eye syndrome|d...,pilocarpine,Pilocarpine,C0031923,Pilocarpine,-1,-1,C0031923,Pilocarpine,sjogren syndrome
33531,NCT00637793,Sjogren's Syndrome | Xerostomia | decreased sa...,MONDO:0010030|-1|-1|MONDO:0010030,Sjogren syndrome|Xerostomia|decreased salivary...,Sjogren syndrome|Xerostomia|decreased salivary...,MONDO:0010030|-1|-1,-1,-1,MONDO:0010030|-1|-1,Sjogren syndrome|Xerostomia|decreased salivary...,ngx267,NGX267,C4282096,NGX267,-1,-1,C4282096,NGX267,sjogren syndrome
42340,NCT00852839,Dry Mouth Associated With Sjogren's Syndrome |...,-1|-1|-1|MONDO:0010030|MONDO:0010030,Dry Mouth Associated With Sjogren's Syndrome|X...,Dry Mouth Associated With Sjogren's Syndrome|X...,-1|-1|-1|MONDO:0010030,-1,-1,-1|-1|-1|MONDO:0010030,Dry Mouth Associated With Sjogren's Syndrome|X...,552-02,552-02,-1,552-02,-1,-1,-1,552-02,sjogren syndrome
48710,NCT00928161,Gastroesophageal Reflux Disease | Head | Neck ...,MONDO:0007186|-1|MONDO:0021310|MONDO:0004608|M...,gastroesophageal reflux disease|Head|malignant...,gastroesophageal reflux disease|Head|malignant...,MONDO:0007186|-1|MONDO:0021310|MONDO:0004608|-...,-1|-1|-1|-1|-1,-1|-1|-1|-1|-1,MONDO:0007186|-1|MONDO:0021310|MONDO:0004608|-...,gastroesophageal reflux disease|Head|malignant...,Dexlansoprazole,Dexlansoprazole,C2348248,Dexlansoprazole,-1,-1,C2348248,Dexlansoprazole,olivopontocerebellar atrophy
60415,NCT01369589,Sjo

In [64]:
clinical_filtered = clinical_df[
    clinical_df["nct_id"].isin(clinical_neuro["nct_id"])
]

In [65]:
clinical_filtered.nct_id.nunique()

48095

In [66]:
disease_terms.str.contains("infectious disease", case=False, na=False).any()

True

In [74]:
matches = disease_terms[disease_terms.str.contains("xerostomia", case=False, na=False)]
print(matches)

Series([], Name: disease_mondo_term_norm, dtype: object)


In [69]:
clinical_filtered.to_csv("/scratch/sdonev/Clinical_Pipeline/data/linked_to_ontologies/entities_drug_disease_clin_neuro.csv",index=False)

### filter metadata

In [70]:
clinical_metadata = pd.read_csv("./data/raw_aact/mv_interventional_drug_studies_20260302.csv")
clinical_metadata.shape

(293949, 12)

In [71]:
clinical_metadata_filtered = clinical_metadata[
    clinical_metadata["nct_id"].isin(clinical_neuro["nct_id"])
]
clinical_metadata_filtered.shape

(48095, 12)

In [73]:
clinical_metadata_filtered.to_csv("./data/raw_aact/mv_interventional_drug_studies_neuro.csv",index=False)

### preclinical

In [ ]:
FILE_PRECLINICAL_LINKING = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/entities_drug_disease_preclin.csv"
preclinical_df = pd.read_csv(FILE_PRECLINICAL_LINKING)


In [ ]:
# Split and explode conditions and drugs
#preclinical_df[conditions_col_to_use] = preclinical_df[conditions_col_to_use].str.split("|")
#preclinical_df = preclinical_df.explode(conditions_col_to_use, ignore_index=True)

# DISEASE
cols_to_explode = [
    conditions_col_to_use,
    "merged_mondo_termid"
    
]

for col in cols_to_explode:
    preclinical_df[col] = preclinical_df[col].astype(str).str.split("|")

preclinical_df = preclinical_df.explode(cols_to_explode, ignore_index=True)
print(f"Unique disease before length filter: {preclinical_df[conditions_col_to_use].nunique()}")
#preclinical_df = preclinical_df[preclinical_df[conditions_col_to_use].fillna("").str.len() > 2]
#print(f"Unique disease: {preclinical_df[conditions_col_to_use].nunique()}")

# DRUG
cols_to_explode = [
    drugs_col_to_use,     # e.g. drug names
    "merged_umls_termid",             # IDs
    
]

for col in cols_to_explode:
    preclinical_df[col] = preclinical_df[col].astype(str).str.split("|")

preclinical_df = preclinical_df.explode(cols_to_explode, ignore_index=True)
print(f"Unique drugs before length filter: {preclinical_df[drugs_col_to_use].nunique()}")

preclinical_df = preclinical_df[preclinical_df[drugs_col_to_use].fillna("").str.len() > 2]
print(f"Unique drugs: {preclinical_df[drugs_col_to_use].nunique()}")

preclinical_df = preclinical_df.drop_duplicates()

# Strip whitespace and convert to lowercase
preclinical_df[conditions_col_to_use] = preclinical_df[conditions_col_to_use].str.strip().str.lower()
preclinical_df[drugs_col_to_use] = preclinical_df[drugs_col_to_use].str.strip().str.lower()

In [ ]:
terms = sorted({
    t.strip().lower()
    for t in terms_list
    if isinstance(t, str) and t.strip()
}, key=len, reverse=True)

# Prepare text column once
s = (
    preclinical_df["merged_mondo_label"]
    .fillna("")
    .astype(str)
    .str.lower()
)

mask = np.zeros(len(s), dtype=bool)

chunk_size = 500  # adjust if needed

for i in tqdm(range(0, len(terms), chunk_size)):
    chunk_terms = terms[i:i+chunk_size]
    pattern = re.compile("|".join(map(re.escape, chunk_terms)))
    mask |= s.str.contains(pattern, na=False).to_numpy()

preclinical_neuro = preclinical_df[mask]

preclinical_neuro.shape

In [ ]:
preclinical_df = pd.read_csv(FILE_PRECLINICAL_LINKING)
preclinical_data_filtered = preclinical_df[
    preclinical_df["PMID"].isin(preclinical_neuro["PMID"])
]
preclinical_data_filtered.shape

In [ ]:
preclinical_data_filtered.to_csv('/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/entities_drug_disease_preclin_neuro.csv', index=False)